<a href="https://colab.research.google.com/github/zoha200/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zoha200/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring
**Data:** `data/raw/content_refresh_anonymized.csv` (starter set, 30,000 rows — the same slice used in ML-02/ML-03)
**Outcome used only to check signals (never fed into the rule as an input):** `is_declining_proxy` from ML-03 — `impressions_last_30d < impressions_prev_30d`, an observed trailing-window comparison, not a future window and not derived from `trend_direction`/`trend_pct`.


In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zoha200/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_proxy"] = (df["impressions_last_30d"] < df["impressions_prev_30d"]).astype(int)
print(f"{len(df)} rows loaded | overall decline rate: {df['is_declining_proxy'].mean():.3f}")


30000 rows loaded | overall decline rate: 0.657


## 1. Two signal checks, then my rule and its reason codes

**Signal 1 — staleness, behind FlyRank's refresh flags.** The idea a refresh flag leans on: a page
that hasn't been touched in a long time is more likely to be declining. I bucket `days_since_last_update`
using the dataset's own `freshness_tier` (0-30 / 31-90 / 91-180 / 181+) and check the decline rate in
each bucket, with `n` printed per bucket.

**Signal 2 — CTR-vs-position, behind the CTR-fix logic.** The idea behind a CTR-fix flag: among pages
that are actually visible (enough impressions to matter) and have real position data, a page pulling a
weak click-through rate for its traffic level is more likely to be declining than one with a healthy CTR.
I restrict to visible pages (`avg_position > 0` and `impressions_90d >= 100`, so "no data" rows and
barely-seen pages don't distort the read), bucket CTR into zero / low / mid / high, and check the decline
rate in each bucket, with `n` printed per bucket.


In [2]:
# --- Signal 1: staleness (freshness_tier) vs decline rate ---
order = ["0-30", "31-90", "91-180", "181+"]
signal1 = (
    df.groupby("freshness_tier", observed=True)
    .agg(n=("is_declining_proxy", "size"), decline_rate=("is_declining_proxy", "mean"))
    .reindex(order)
)
print("Signal 1 — staleness (freshness_tier) bucket table:")
print(signal1.round(3))
print()
print(f"Overall decline rate for reference: {df['is_declining_proxy'].mean():.3f} (n={len(df)})")


Signal 1 — staleness (freshness_tier) bucket table:
                    n  decline_rate
freshness_tier                     
0-30            20480         0.614
31-90             175         0.657
91-180           9171         0.756
181+              174         0.517

Overall decline rate for reference: 0.657 (n=30000)


**Verdict — Signal 1 (staleness): MIXED.**
Decline rate rises from 61.4% (0-30d, n=20,480) to 65.7% (31-90d, n=175) to 75.9% (91-180d, n=9,171) —
directionally right, staler pages decline more. But the 181+ tier drops back to 51.7% (n=174). That
last tier is thin (under 1% of the data) so I don't fully trust it, but I can't just discard an
inconvenient result either — an honest verdict is MIXED, not CONFIRMED. Staleness alone is not a clean
enough signal to carry a rule by itself; it belongs as a secondary factor, not the primary one.


In [3]:
# --- Signal 2: CTR-vs-position among visible pages with real position data ---
visible = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 100)].copy()

def ctr_bucket(x):
    if x == 0:
        return "zero_ctr"
    elif x < 0.14:
        return "low_ctr"
    elif x < 0.34:
        return "mid_ctr"
    else:
        return "high_ctr"

visible["ctr_bucket"] = visible["ctr"].apply(ctr_bucket)
order2 = ["zero_ctr", "low_ctr", "mid_ctr", "high_ctr"]
signal2 = (
    visible.groupby("ctr_bucket", observed=True)
    .agg(n=("is_declining_proxy", "size"), decline_rate=("is_declining_proxy", "mean"), avg_position=("avg_position", "mean"))
    .reindex(order2)
)
print(f"Visible-page slice: n={len(visible)} (avg_position > 0 and impressions_90d >= 100)")
print("Signal 2 — CTR-vs-position bucket table:")
print(signal2.round(3))


Visible-page slice: n=22006 (avg_position > 0 and impressions_90d >= 100)
Signal 2 — CTR-vs-position bucket table:
               n  decline_rate  avg_position
ctr_bucket                                  
zero_ctr    6076         0.723        24.880
low_ctr     4546         0.807        18.468
mid_ctr     5781         0.750        13.994
high_ctr    5603         0.715        11.584


**Verdict — Signal 2 (CTR-vs-position): CONFIRMED.**
Decline rate falls as CTR rises: zero_ctr 72.3% (n=6,076) and low_ctr 80.7% (n=4,546), down to mid_ctr
75.0% (n=5,781) and high_ctr 71.5% (n=5,603, best average position at 11.6). The core idea — weak CTR
for the traffic/position a page has predicts decline — holds. `zero_ctr` sits a bit below `low_ctr`
rather than lowest overall, which makes sense: a true zero often means too few clicks to register
rather than a genuinely worse page, so it's a slightly different case than a real-but-low CTR. That
nuance doesn't break the confirmation, it just means the rule should treat `low_ctr` (not `zero_ctr`)
as the sharpest trigger.

**My rule, in plain words:** a page is worth reviewing first if it's visible enough to matter, and
either its click-through rate is weak for the position it holds, or it hasn't been touched in a long
time — and it gets flagged hardest when both are true at once. CTR-vs-position leads because it was the
cleaner, CONFIRMED signal; staleness rides along as a secondary factor because it was only MIXED.

**Reason codes (exactly one per row, in priority order):**
1. `stale_and_ctr_gap` — visible, low/zero CTR for its position, AND stale (91d+)
2. `ctr_gap_visible` — visible, low/zero CTR for its position, not stale
3. `stale_visible` — visible and stale, but CTR is healthy
4. `low_priority_review` — none of the above triggers fired

**Action labels:** `stale_and_ctr_gap` → refresh_and_fix_ctr · `ctr_gap_visible` → review_ctr_and_meta ·
`stale_visible` → refresh · `low_priority_review` → monitor


In [4]:
# --- The rule: score + ONE reason code + action label ---
def score_row(row):
    is_visible = row["impressions_90d"] >= 100
    has_position = row["avg_position"] > 0
    is_low_ctr = has_position and row["ctr"] < 0.34          # mid_ctr and below, on a real position
    is_stale = row["days_since_last_update"] >= 91            # freshness_tier 91-180 or 181+

    if is_visible and is_low_ctr and is_stale:
        reason_code = "stale_and_ctr_gap"
        action = "refresh_and_fix_ctr"
        weight = 1.0
    elif is_visible and is_low_ctr:
        reason_code = "ctr_gap_visible"
        action = "review_ctr_and_meta"
        weight = 0.7
    elif is_visible and is_stale:
        reason_code = "stale_visible"
        action = "refresh"
        weight = 0.4
    else:
        reason_code = "low_priority_review"
        action = "monitor"
        weight = 0.05

    # Transparent score: trigger weight, scaled by how much visibility is actually at stake.
    score = weight * np.log1p(row["impressions_90d"])
    return pd.Series({"score": score, "reason_code": reason_code, "action": action})

scored = df.apply(score_row, axis=1)
print(scored["reason_code"].value_counts())
print()
print(scored.head(3))


reason_code
low_priority_review    11698
ctr_gap_visible        10183
stale_and_ctr_gap       6220
stale_visible           1899
Name: count, dtype: int64

      score          reason_code               action
0  0.412190  low_priority_review              monitor
1  6.745886      ctr_gap_visible  review_ctr_and_meta
2  6.608016      ctr_gap_visible  review_ctr_and_meta


## 2. Build the ranked queue (writes the CSV)


In [5]:
out = pd.concat([
    df[["content_id", "client_id", "impressions_90d", "days_since_last_update",
        "ctr", "avg_position", "content_age_days", "is_declining_proxy"]],
    scored
], axis=1)

out = out.sort_values("score", ascending=False).reset_index(drop=True)
out.insert(0, "rank", out.index + 1)

os.makedirs("work/outputs", exist_ok=True)
out.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(out)} ranked rows to work/outputs/baseline_action_score.csv")
print(out.head(10)[["rank", "content_id", "score", "reason_code", "action"]])


Wrote 30000 ranked rows to work/outputs/baseline_action_score.csv
   rank            content_id      score        reason_code  \
0     1  content_5fe46e04994d  13.157182  stale_and_ctr_gap   
1     2  content_2dba2b1f9536  13.002307  stale_and_ctr_gap   
2     3  content_cb112fce36be  12.644040  stale_and_ctr_gap   
3     4  content_36ff89c8214e  12.595063  stale_and_ctr_gap   
4     5  content_b28d1efd668f  12.565874  stale_and_ctr_gap   
5     6  content_813e88069237  12.361203  stale_and_ctr_gap   
6     7  content_c8e9d6ab9013  12.248552  stale_and_ctr_gap   
7     8  content_b511d4bc4ad2  12.235224  stale_and_ctr_gap   
8     9  content_d17681677e69  12.213966  stale_and_ctr_gap   
9    10  content_a7427266c305  12.211617  stale_and_ctr_gap   

                action  
0  refresh_and_fix_ctr  
1  refresh_and_fix_ctr  
2  refresh_and_fix_ctr  
3  refresh_and_fix_ctr  
4  refresh_and_fix_ctr  
5  refresh_and_fix_ctr  
6  refresh_and_fix_ctr  
7  refresh_and_fix_ctr  
8  refresh_and_

## 3. Top-10 review


In [6]:
top10 = out.head(10)
for _, r in top10.iterrows():
    print(f"#{r['rank']:>2} | action={r['action']:<20} | score={r['score']:.2f}")
    print(f"     content_id={r['content_id']}, client_id={r['client_id']}")
    print(f"     impressions_90d={r['impressions_90d']:.0f}, ctr={r['ctr']:.2f}, "
          f"avg_position={r['avg_position']:.1f}, days_since_last_update={r['days_since_last_update']:.0f}, "
          f"reason={r['reason_code']}")
    print()


# 1 | action=refresh_and_fix_ctr  | score=13.16
     content_id=content_5fe46e04994d, client_id=client_4e07408562
     impressions_90d=517715, ctr=0.14, avg_position=4.2, days_since_last_update=104, reason=stale_and_ctr_gap

# 2 | action=refresh_and_fix_ctr  | score=13.00
     content_id=content_2dba2b1f9536, client_id=client_6208ef0f77
     impressions_90d=443434, ctr=0.21, avg_position=27.9, days_since_last_update=104, reason=stale_and_ctr_gap

# 3 | action=refresh_and_fix_ctr  | score=12.64
     content_id=content_cb112fce36be, client_id=client_19581e27de
     impressions_90d=309910, ctr=0.16, avg_position=5.6, days_since_last_update=104, reason=stale_and_ctr_gap

# 4 | action=refresh_and_fix_ctr  | score=12.60
     content_id=content_36ff89c8214e, client_id=client_19581e27de
     impressions_90d=295097, ctr=0.05, avg_position=7.3, days_since_last_update=104, reason=stale_and_ctr_gap

# 5 | action=refresh_and_fix_ctr  | score=12.57
     content_id=content_b28d1efd668f, client_id=cli

**Row-by-row: why it's there, and what would make it wrong**

All ten top-ranked rows share the same trigger (`stale_and_ctr_gap` → `refresh_and_fix_ctr`) and the
exact same `days_since_last_update` of 104 — a pattern worth naming honestly below, not hiding.
What separates them is scale (impressions_90d) and how weak the CTR-vs-position gap actually is.

*(rank — why it's there — what would make it wrong)*

1. **517,715 impressions, ctr=0.14 at position 4.2** — huge audience, strong position, still a weak
   click rate — the clearest case of "the content or snippet isn't earning the clicks its ranking
   deserves." **Wrong if:** the SERP has a competing feature (image pack, PAA box) suppressing clicks
   regardless of this page's quality.
2. **443,434 impressions, ctr=0.21 at position 27.9** — `is_declining_proxy=0` (not actually declining
   by the 30-day proxy) despite firing the rule's strongest trigger. **This is a real disagreement**,
   not just a hypothetical — see Section 4.
3. **309,910 impressions, ctr=0.16 at position 5.6** — good position, real traffic, CTR still low for
   that position. **Wrong if:** this is a highly branded query where users already know what's behind
   the link and click at their own, lower, baseline rate.
4. **295,097 impressions, ctr=0.05 at position 7.3** — `is_declining_proxy=0` — the second real
   disagreement in the top 10 (see Section 4).
5. **286,608 impressions, ctr=0.06 at position 26.2** — weak position AND weak CTR together, but still
   high volume. **Wrong if:** the query intent behind these impressions doesn't match this page at all
   (a mismatched-intent ranking), which a refresh wouldn't fix — it needs a different target query.
6. **233,561 impressions, ctr=0.06 at position 26.2** — same client and pattern as #5. **Wrong if:**
   it's a genuine duplicate/near-duplicate of #5 competing for the same query, in which case refreshing
   both is wasted effort — one should be consolidated or redirected instead.
7. **208,678 impressions, ctr=0.00 at position 9.7** — the only true zero-CTR row in the top 10, on a
   decent position. **Wrong if:** `ctr=0.00` here means "not enough clicks to register" rather than
   "nobody clicks it" — i.e. a measurement artifact more than a content problem.
8. **205,915 impressions, ctr=0.14 at position 27.9** — same client and position band as #2. **Wrong
   if:** this is one of several near-identical listings from the same client, and the real fix is
   deduping rather than refreshing each one individually.
9. **201,584 impressions, ctr=0.24 at position 5.8** — the healthiest CTR in the top 10, good position,
   still flagged mainly because of the 104-day staleness. **Wrong if:** the page was substantively
   updated recently and `days_since_last_update` is stale metadata, not a stale page.
10. **201,111 impressions, ctr=0.11 at position 5.7** — right at the bottom of the top 10 on score, but
    still a strong position with a middling CTR. **Wrong if:** this position/CTR combination is just
    normal variance for this content type, not an actual opportunity.


## 4. Weak picks + leakage check


In [7]:
# A weak pick should be one the rule ranked highly but the observed proxy disagrees with.
weak = out.head(10)[out.head(10)["is_declining_proxy"] == 0]
print(f"{len(weak)} of the top 10 are flagged 'refresh_and_fix_ctr' but is_declining_proxy == 0")
print("(the rule's strongest trigger fired, but the 30-day proxy says this page is NOT currently declining):")
print(weak[["rank", "content_id", "impressions_90d", "ctr", "avg_position",
            "days_since_last_update", "reason_code", "action", "is_declining_proxy"]].to_string(index=False))


2 of the top 10 are flagged 'refresh_and_fix_ctr' but is_declining_proxy == 0
(the rule's strongest trigger fired, but the 30-day proxy says this page is NOT currently declining):
 rank           content_id  impressions_90d  ctr  avg_position  days_since_last_update       reason_code              action  is_declining_proxy
    2 content_2dba2b1f9536           443434 0.21          27.9                     104 stale_and_ctr_gap refresh_and_fix_ctr                   0
    4 content_36ff89c8214e           295097 0.05           7.3                     104 stale_and_ctr_gap refresh_and_fix_ctr                   0


**Which picks look weakest, and why:** ranks 2 and 4 — both fire the rule's strongest trigger
(`stale_and_ctr_gap`) on real traffic (443k and 295k impressions), yet `is_declining_proxy == 0` for
both: by the observed 30-day comparison, these pages are not currently declining. That's the honest
failure mode of a CTR-vs-position rule — a page can hold a structurally low CTR for its position
(intent mismatch, a branded query, a SERP feature) without actually losing traffic month over month.
The rule can't tell "weak CTR that predicts decline" apart from "weak CTR that's just this page's
steady state," which is exactly the kind of case a learned model should be able to separate and this
hand-written rule cannot.

**A second honest observation, not asked for but worth naming:** all ten top-ranked rows share the
identical `days_since_last_update` of 104. That's a real pattern in this slice, not a coding error —
likely a handful of clients whose pages were bulk-updated on the same date, which also happen to be
high-traffic. It's a reminder that `days_since_last_update` alone won't cleanly separate individually
neglected pages from a client's entire catalog being touched at once — another reason staleness came
back MIXED rather than CONFIRMED in Section 1.

**Leakage check — confirmed clean:**
- The score uses only `impressions_90d`, `ctr`, `avg_position`, and `days_since_last_update` — all
  observed as of now, none derived from a future window.
- `trend_direction` and `trend_pct` (the label-derived columns flagged back in ML-02/ML-03) are **not**
  used anywhere in `score_row`.
- `is_declining_proxy` is carried in the output CSV only as a reference column to check the rule
  against — it is never read inside `score_row`, so it never influences the score, reason code, or
  action. Ranks 2 and 4 above are exactly what that separation is for: it let a real disagreement
  surface instead of being scored away.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
